In [ ]:
# 100% compatible (Colab & VS Code)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Environment: Google Colab (Drive mounted)")
except ImportError:
    print("Environment: Local (VS Code)")

# Milestone 1: MNIST Data Exploration & Classical Baseline

**Objectives:**
1. Configure the runtime environment and connect **MLflow / DagsHub** experiment tracking.
2. Load the **MNIST** dataset, verify class balance, and visually inspect sample images.
3. Normalize pixel values and perform a reproducible **54,000 train / 6,000 validation** split (keeping the 10,000-image test set strictly untouched).
4. Train a **scikit-learn** baseline (Logistic Regression), measure validation error, inspect the confusion matrix, and log metrics and artifacts to DagsHub.

In [ ]:
import os
import platform
import time

import dagshub
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import sklearn
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras

### 1 - Reproducibility & DagsHub (MLflow) Connection

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Log library versions to ensure reproducibility
print(f"Python       : {platform.python_version()}")
print(f"NumPy        : {np.__version__}")
print(f"Scikit-learn : {sklearn.__version__}")
print(f"TensorFlow   : {tf.__version__}")
print(f"MLflow       : {mlflow.__version__}")

In [ ]:
REPO_OWNER = "bruno-coulet"
REPO_NAME = "handwritten_digits"

# Authenticate with DagsHub and configure MLflow tracking URI
dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)
mlflow.set_experiment("Milestone_1_Baseline")

### 2 - Loading & Exploring the MNIST Dataset

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Full train set :", x_train_full.shape, y_train_full.shape)
print("Test set (held):", x_test.shape, y_test.shape)
print("Data type      :", x_train_full.dtype, "| min/max:", x_train_full.min(), x_train_full.max())

In [ ]:
# Display a grid of random samples per digit class (0 to 9)
fig, axes = plt.subplots(10, 8, figsize=(10, 12))
for digit in range(10):
    digit_indices = np.where(y_train_full == digit)[0]
    chosen = np.random.choice(digit_indices, size=8, replace=False)
    for col, idx in enumerate(chosen):
        axes[digit, col].imshow(x_train_full[idx], cmap="gray")
        axes[digit, col].axis("off")
        if col == 0:
            axes[digit, col].set_title(f"Class {digit}", loc="left", fontsize=10, fontweight="bold")

plt.suptitle("MNIST Sample Grid by Class (0 to 9)", fontsize=14, y=0.92)
plt.tight_layout()
plt.show()

In [ ]:
# Check class distribution across the 60,000 training images
classes, counts = np.unique(y_train_full, return_counts=True)

plt.figure(figsize=(8, 4))
bars = plt.bar(classes, counts, color="#2b5c8f")
plt.xticks(classes)
plt.xlabel("Digit Class")
plt.ylabel("Number of Images")
plt.title("Class Distribution in Training Set (60,000 images)")
for bar, count in zip(bars, counts, strict=True):
    plt.text(bar.get_x() + bar.get_width() / 2, count + 100, str(count), ha="center", fontsize=9)
plt.ylim(0, max(counts) * 1.1)
plt.show()

In [ ]:
# Compute and plot the mean image per class
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for c in range(10):
    axes[c].imshow(x_train_full[y_train_full == c].mean(axis=0), cmap="gray")
    axes[c].set_title(str(c))
    axes[c].axis("off")
plt.suptitle("Mean Image per Digit Class", y=1.05)
plt.show()

In [ ]:
# Inspect raw pixel intensity values on a single sample
sample_idx = 8

plt.figure(figsize=(7, 7))
plt.imshow(x_train_full[sample_idx], cmap="gray")
for i in range(28):
    for j in range(28):
        v = x_train_full[sample_idx][i, j]
        if v > 0:
            plt.text(
                j, i, v, ha="center", va="center", fontsize=6,
                color="black" if v > 128 else "white"
            )
plt.title(f"Raw Pixel Intensities (Label: {y_train_full[sample_idx]})")
plt.axis("off")
plt.show()

### 3 - Preprocessing & Train / Validation / Test Split

* **Normalization**: Scale pixel values from `[0, 255]` to `[0.0, 1.0]` (`float32`).
* **Flattening**: Reshape `28 x 28` images into 1D feature vectors of length `784` for scikit-learn.
* **Stratified Split**: `54,000` training samples and `6,000` validation samples. The `10,000`-image test set remains strictly untouched during Milestone 1.

In [ ]:
# Normalize pixel intensities to [0, 1]
x_train_norm = x_train_full.astype("float32") / 255.0
x_test_norm = x_test.astype("float32") / 255.0

# Stratified split: 54,000 train / 6,000 validation
X_train_img, X_val_img, y_train, y_val = train_test_split(
    x_train_norm,
    y_train_full,
    test_size=6000,
    random_state=SEED,
    stratify=y_train_full,
)

# Flatten images to (N, 784) for classical ML models
X_train = X_train_img.reshape(len(X_train_img), -1)
X_val = X_val_img.reshape(len(X_val_img), -1)
X_test = x_test_norm.reshape(len(x_test_norm), -1)

print(f"X_train : {X_train.shape} | y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}   | y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}  | y_test  : {y_test.shape} (untouched in Milestone 1)")

### 4 - Baseline Training (Logistic Regression) & DagsHub Logging

In [ ]:
params = {
    "model_type": "LogisticRegression",
    "max_iter": 1000,
    "solver": "lbfgs",
    "C": 1.0,
    "seed": SEED,
    "train_size": len(X_train),
    "val_size": len(X_val),
    "scaling": "min_max_0_1",
}

with mlflow.start_run(run_name="baseline_logistic_regression"):
    # 1. Log hyperparameters and reproducibility metadata
    mlflow.log_params(params)
    mlflow.set_tag("milestone", "1_baseline")
    mlflow.set_tag("test_set_used", "False")

    # 2. Fit the baseline model
    start_time = time.time()
    clf = LogisticRegression(
        max_iter=params["max_iter"],
        solver=params["solver"],
        C=params["C"],
        random_state=SEED,
    )
    clf.fit(X_train, y_train)
    train_duration = time.time() - start_time

    # 3. Evaluate strictly on the validation set (6,000 images)
    val_preds = clf.predict(X_val)
    val_accuracy = (val_preds == y_val).mean()
    val_error_pct = (1.0 - val_accuracy) * 100

    mlflow.log_metric("val_accuracy", val_accuracy)
    mlflow.log_metric("val_error_pct", val_error_pct)
    mlflow.log_metric("training_duration_sec", train_duration)

    print(f"Training duration   : {train_duration:.2f} s")
    print(f"Validation Accuracy : {val_accuracy:.4f}")
    print(f"Validation Error (%): {val_error_pct:.2f} %")

    # 4. Generate and log confusion matrix artifact
    cm = confusion_matrix(y_val, val_preds)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    plt.title(f"Confusion Matrix - Baseline LogReg (Val Error: {val_error_pct:.2f}%)")
    plt.tight_layout()

    os.makedirs("reports/figures", exist_ok=True)
    cm_path = "reports/figures/milestone1_baseline_confusion_matrix.png"
    plt.savefig(cm_path, dpi=150)
    plt.show()

    mlflow.log_artifact(cm_path)
    mlflow.sklearn.log_model(clf, artifact_path="baseline_model")

### 5 - Baseline Error Analysis (Most Confused Pairs)

In [ ]:
# Extract top 10 most frequent misclassifications (True -> Predicted)
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

flat_indices = np.argsort(cm_no_diag.ravel())[::-1][:10]
print("Top 10 misclassified pairs on the validation set (6,000 images):\n")
for idx in flat_indices:
    true_label, pred_label = divmod(idx, 10)
    count = cm_no_diag[true_label, pred_label]
    print(f"  True: {true_label}  -->  Predicted: {pred_label}  ({count} errors)")

print("\nDetailed Classification Report (Validation Set):\n")
print(classification_report(y_val, val_preds, digits=4))

#### Interpretation & Hypothesis
* **Validation Error Rate**: ~7.5% to 8.0% error on the validation set (~92% accuracy).
* **Most Confused Pairs**: The highest confusion rates typically occur between **(4 and 9)**, **(3 and 5)**, **(5 and 8)**, and **(7 and 9)**.
* **Hypothesis**: Logistic Regression is a linear classifier that assigns an independent weight to each pixel coordinate `(i, j)` without spatial awareness. Digits such as a closed-top `4` and `9`, or `3` and `5`, share a large overlap of active pixels in the center of the $28 \times 28$ grid and differ only by localized loops or stroke angles.